# Plant Dataset — Decoy Strategy Comparison
Comparison of 4 decoy strategies (score_coord, score_coord_noise, nearest_neighbor, nearest_neighbor_noise) for accuracy estimation via Mix-Max FDR on the plant-daniella dataset.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import os
import tarfile
import io

print(f"NumPy {np.__version__}  PyTorch {torch.__version__}")


## Configuration

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

plt.rcParams.update({
    'font.size': 13, 'axes.titlesize': 14, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11, 'legend.fontsize': 9,
    'figure.titlesize': 15, 'font.family': 'serif',
    'figure.dpi': 100, 'savefig.dpi': 150, 'savefig.bbox': 'tight',
})

os.makedirs('figures', exist_ok=True)


## Utility functions (calibration, baselines)

In [ ]:
def negentropy(logits):
    if isinstance(logits, np.ndarray):
        logits = torch.from_numpy(logits).float()
    probs = torch.softmax(logits, dim=1)
    entropy = -(probs * torch.log(probs + 1e-10)).sum(dim=1)
    return np.log(logits.shape[1]) - entropy

def calibration_temp(logits, labels, num_bins=15):
    if isinstance(logits, np.ndarray): logits = torch.from_numpy(logits).float()
    if isinstance(labels, np.ndarray): labels = torch.from_numpy(labels).long()
    temps = torch.linspace(0.1, 5.0, 50)
    best_temp, best_ece = 1.0, float('inf')
    for temp in temps:
        p = torch.softmax(logits / temp, dim=1)
        conf, pred = p.max(1); acc = (pred == labels).float()
        bins = torch.linspace(0, 1, num_bins + 1)
        ece = sum(
            ((conf > bins[i]) & (conf <= bins[i+1])).float().mean() *
            abs(conf[(conf > bins[i]) & (conf <= bins[i+1])].mean() -
                acc[(conf > bins[i]) & (conf <= bins[i+1])].mean()).item()
            for i in range(num_bins)
            if ((conf > bins[i]) & (conf <= bins[i+1])).sum() > 0
        )
        if ece < best_ece: best_ece, best_temp = ece, temp.item()
    return best_temp

def _to_tensor(x):
    return torch.from_numpy(x).float() if isinstance(x, np.ndarray) else x

def predict_ATC_maxconf(src_logits, src_labels, tgt_logits):
    src_logits = _to_tensor(src_logits); src_labels = _to_tensor(src_labels).long()
    tgt_logits = _to_tensor(tgt_logits)
    src_sc = torch.softmax(src_logits, 1).amax(1)
    tgt_sc = torch.softmax(tgt_logits, 1).amax(1)
    thr = torch.sort(src_sc).values[-(src_logits.argmax(1) == src_labels).sum()]
    return (tgt_sc > thr).float().mean().item()

def predict_ATC_negent(src_logits, src_labels, tgt_logits):
    src_logits = _to_tensor(src_logits); src_labels = _to_tensor(src_labels).long()
    tgt_logits = _to_tensor(tgt_logits)
    src_sc = negentropy(src_logits); tgt_sc = negentropy(tgt_logits)
    thr = torch.sort(src_sc).values[-(src_logits.argmax(1) == src_labels).sum()]
    return (tgt_sc > thr).float().mean().item()

def predict_AC(src_logits, src_labels, tgt_logits):
    return torch.softmax(_to_tensor(tgt_logits), 1).amax(1).mean().item()

def predict_DOC(src_logits, src_labels, tgt_logits):
    sl = _to_tensor(src_logits); sb = _to_tensor(src_labels).long(); tl = _to_tensor(tgt_logits)
    src_conf = torch.softmax(sl, 1).amax(1).mean().item()
    tgt_conf = torch.softmax(tl, 1).amax(1).mean().item()
    src_acc  = (sl.argmax(1) == sb).float().mean().item()
    return src_acc + (tgt_conf - src_conf)

BASELINE_METHODS = {'ATC': predict_ATC_maxconf, 'ATC-NE': predict_ATC_negent,
                    'AC': predict_AC, 'DOC': predict_DOC}

try:
    import ot
    def predict_COT(sl, sb, tl):
        sl = _to_tensor(sl); sb = _to_tensor(sb).long(); tl = _to_tensor(tl)
        nc = sl.shape[1]
        lbl_dist = F.one_hot(sb, nc).float().mean(0)
        tp = torch.softmax(tl, 1)
        cost = torch.stack([(tp - F.one_hot(torch.tensor(k), nc).float()).abs().sum(1) / 2 for k in range(nc)], 1)
        ot_plan = ot.emd(np.ones(len(tp)) / len(tp), lbl_dist.numpy(), cost.numpy())
        ot_cost = (ot_plan * cost.numpy()).sum()
        return 1 - (ot_cost + torch.softmax(sl, 1).amax(1).mean().item() - (sl.argmax(1) == sb).float().mean().item())
    BASELINE_METHODS['COT'] = predict_COT
    print("COT available")
except ImportError:
    print("COT unavailable (pip install POT)")
print(f"Baseline methods: {list(BASELINE_METHODS.keys())}")


## Flow model — building blocks

In [ ]:
class RobustFeatureNormalizer(nn.Module):
    def __init__(self, feature_dim, clip_val=5.0, momentum=0.01, eps=1e-6):
        super().__init__()
        self.clip_val = clip_val; self.momentum = momentum; self.eps = eps
        self.register_buffer('running_median', torch.zeros(feature_dim))
        self.register_buffer('running_iqr',    torch.ones(feature_dim))
        self.register_buffer('initialized',    torch.tensor(False))

    @torch.no_grad()
    def _update_stats(self, x):
        bm = x.median(0).values
        bi = (torch.quantile(x, 0.75, 0) - torch.quantile(x, 0.25, 0)).clamp(min=self.eps)
        if not self.initialized:
            self.running_median.copy_(bm); self.running_iqr.copy_(bi)
            self.initialized.fill_(True)
        else:
            self.running_median.mul_(1 - self.momentum).add_(bm * self.momentum)
            self.running_iqr.mul_(1 - self.momentum).add_(bi * self.momentum)

    def forward(self, x):
        if self.training: self._update_stats(x)
        if not self.initialized: return torch.tanh(x * 0.01)
        xn = ((x - self.running_median) / (self.running_iqr + self.eps)).clamp(-self.clip_val, self.clip_val)
        return torch.tanh(xn / self.clip_val)


class ActNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.log_scale = nn.Parameter(torch.zeros(dim))
        self.bias      = nn.Parameter(torch.zeros(dim))
        self.register_buffer('initialized', torch.tensor(False))

    def forward(self, x, reverse=False):
        if not self.initialized and not reverse:
            with torch.no_grad():
                self.bias.data      = -x.mean(0)
                self.log_scale.data = -x.std(0).clamp(min=1e-6).log()
            self.initialized.fill_(True)
        if not reverse:
            return (x + self.bias) * self.log_scale.exp(), self.log_scale.sum().expand(x.size(0))
        return x * (-self.log_scale).exp() - self.bias, -self.log_scale.sum().expand(x.size(0))


class CouplingLayer(nn.Module):
    def __init__(self, dim, feature_dim, hidden_dim=256, mask_type='first_half'):
        super().__init__()
        self.mask_type = mask_type
        self.d_in  = dim // 2 if mask_type == 'first_half' else dim - dim // 2
        self.d_out = dim - dim // 2 if mask_type == 'first_half' else dim // 2
        self.net = nn.Sequential(
            nn.Linear(self.d_in + feature_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU())
        self.scale_head     = nn.Sequential(nn.Linear(hidden_dim // 2, self.d_out), nn.Tanh())
        self.translate_head = nn.Linear(hidden_dim // 2, self.d_out)
        nn.init.zeros_(self.scale_head[0].weight); nn.init.zeros_(self.scale_head[0].bias)
        nn.init.zeros_(self.translate_head.weight); nn.init.zeros_(self.translate_head.bias)

    def _split(self, x):
        return (x[:, :self.d_in], x[:, self.d_in:]) if self.mask_type == 'first_half'                else (x[:, self.d_out:], x[:, :self.d_out])

    def _merge(self, x1, x2):
        return torch.cat([x1, x2], 1) if self.mask_type == 'first_half' else torch.cat([x2, x1], 1)

    def forward(self, x, features, reverse=False):
        x1, x2 = self._split(x); h = self.net(torch.cat([x1, features], 1))
        s, t = self.scale_head(h), self.translate_head(h)
        if not reverse:
            return self._merge(x1, x2 * torch.exp(s) + t), s.sum(1)
        return self._merge(x1, (x2 - t) * torch.exp(-s)), -s.sum(1)


## ScoreShiftFlow

In [ ]:
class ScoreShiftFlow(nn.Module):
    def __init__(self, score_dim=10, feature_dim=640, n_flows=12, hidden_dim=256,
                 encoder_dim=128, clip_val=5.0):
        super().__init__()
        self.score_dim = score_dim
        self._log_2pi  = float(np.log(2 * np.pi))
        self.feature_norm    = RobustFeatureNormalizer(feature_dim, clip_val=clip_val, momentum=0.01)
        self.feature_encoder = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, encoder_dim), nn.LayerNorm(encoder_dim), nn.GELU())
        self.layers = nn.ModuleList()
        for i in range(n_flows):
            mask = 'first_half' if i % 2 == 0 else 'second_half'
            self.layers.append(CouplingLayer(score_dim, encoder_dim, hidden_dim, mask))
            if i < n_flows - 1:
                self.layers.append(ActNorm(score_dim))

    def encode(self, f): return self.feature_encoder(self.feature_norm(f))

    def forward(self, scores, features, reverse=False):
        enc = self.encode(features)
        ld  = torch.zeros(scores.size(0), device=scores.device)
        if not reverse:
            x = scores
            for layer in self.layers:
                x, d = layer(x, reverse=False) if isinstance(layer, ActNorm)                        else layer(x, enc, reverse=False)
                ld += d
            return x, ld
        z = scores
        for layer in reversed(self.layers):
            z, d = layer(z, reverse=True) if isinstance(layer, ActNorm)                    else layer(z, enc, reverse=True)
            ld += d
        return z, ld

    def log_prob(self, scores, features):
        z, ld = self.forward(scores, features)
        return -0.5 * (z ** 2).sum(1) - 0.5 * self.score_dim * self._log_2pi + ld

    def sample(self, features):
        z = torch.randn(features.size(0), self.score_dim, device=features.device)
        s, _ = self.forward(z, features, reverse=True)
        return s


## ScoreShiftFlowWrapper

In [ ]:
class ScoreShiftFlowWrapper(nn.Module):
    def __init__(self, num_classes=10, n_flows=12, feature_dim=640,
                 hidden_dim=256, encoder_dim=128, clip_val=5.0):
        super().__init__()
        self.num_classes = num_classes
        self.flow = ScoreShiftFlow(num_classes, feature_dim, n_flows, hidden_dim, encoder_dim, clip_val)

    def train_flow(self, score_dataset, epochs=30, lr=3e-4, batch_size=256,
                   device='cuda', patience=5, grad_clip=1.0):
        self.flow.to(device).train()
        loader    = DataLoader(score_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
        optimizer = torch.optim.AdamW(self.flow.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.01)
        best_loss, best_state, no_improve = float('inf'), None, 0
        for epoch in range(epochs):
            total = 0.0; n = 0
            for _, feats, decoys, _ in loader:
                feats = feats.to(device); decoys = decoys.to(device)
                loss = -self.flow.log_prob(decoys, feats).mean()
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(self.flow.parameters(), grad_clip)
                optimizer.step(); total += loss.item(); n += 1
            scheduler.step(); avg = total / max(n, 1)
            if (epoch + 1) % 5 == 0:
                print(f"  Epoch {epoch+1:3d}/{epochs}  loss={avg:.4f}  lr={scheduler.get_last_lr()[0]:.2e}")
            if avg < best_loss - 1e-4:
                best_loss = avg; no_improve = 0
                best_state = {k: v.clone() for k, v in self.flow.state_dict().items()}
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"  Early stop at epoch {epoch+1}")
                    self.flow.load_state_dict(best_state); break
        if best_state: self.flow.load_state_dict(best_state)
        print(f"  Done. Best loss: {best_loss:.4f}")
        return self

    def generate_decoys(self, score_dataset, device='cuda'):
        self.flow.to(device).eval()
        cnn_l, dc_l, lb_l = [], [], []
        loader = DataLoader(score_dataset, batch_size=256, shuffle=False, num_workers=0)
        with torch.no_grad():
            for cnn_sc, feats, _, labels in loader:
                feats = feats.to(device)
                dc_l.append(self.flow.sample(feats).cpu().numpy())
                cnn_l.append(cnn_sc.numpy()); lb_l.append(labels.numpy())
        return np.concatenate(cnn_l), np.concatenate(dc_l), np.concatenate(lb_l)


## ScoreFeatureDataset

In [ ]:
class ScoreFeatureDataset(torch.utils.data.Dataset):
    def __init__(self, cnn_scores, features, target_decoy_scores, labels):
        self.cnn_scores          = cnn_scores
        self.features            = features
        self.target_decoy_scores = target_decoy_scores
        self.labels              = labels
    def __len__(self): return len(self.cnn_scores)
    def __getitem__(self, i):
        return self.cnn_scores[i], self.features[i], self.target_decoy_scores[i], self.labels[i]


## Pool building functions

In [ ]:
MIN_POOL = 30

def build_error_conditioned_pools(train_scores, train_labels, num_classes,
                                   verbose=True, max_pool_size=10_000):
    """pool_score[c] = score_c on examples where argmax=c AND label≠c (fallback: label≠c)."""
    rng = np.random.default_rng(0)
    pred_classes = train_scores.argmax(axis=1)
    pool_score = {}
    if verbose:
        print(f"Building pools  (train acc={(pred_classes == train_labels).mean():.4f})")
    for c in range(num_classes):
        err_mask = (pred_classes == c) & (train_labels != c)
        cands = train_scores[err_mask, c] if err_mask.sum() >= MIN_POOL                 else train_scores[train_labels != c, c]
        pool_score[c] = cands[rng.choice(len(cands), size=max_pool_size, replace=False)]                         if len(cands) > max_pool_size else cands
        if verbose:
            print(f"  class {c}: n={len(pool_score[c]):6d}  [{pool_score[c].min():.3f},{pool_score[c].max():.3f}]")
    return pool_score


def _build_decoy_score_coord(sc_np, pool_score, rng):
    """Replace coord c_hat by a random draw from pool_score[c_hat]."""
    pred = sc_np.argmax(axis=1); dc = sc_np.copy()
    for c in range(sc_np.shape[1]):
        mask = pred == c
        if mask.any() and len(pool_score.get(c, [])) > 0:
            dc[mask, c] = rng.choice(pool_score[c], size=mask.sum(), replace=True)
    return dc


## Data loading utilities

In [ ]:
def load_precomputed_data(path):
    """Load (features, logits, labels) from .pt or .tar.xz archive."""
    if path.endswith(('.tar.xz', '.tar.gz', '.tar.bz2', '.tar')):
        with tarfile.open(path, 'r:*') as tar:
            members = [m for m in tar.getmembers() if m.name.endswith('.pt')]
            if not members: raise FileNotFoundError(f"No .pt inside {path}")
            data = torch.load(io.BytesIO(tar.extractfile(members[0]).read()),
                              map_location='cpu', weights_only=False)
    else:
        data = torch.load(path, map_location='cpu', weights_only=False)
    print(list(data.keys()))
    feat_key  = 'hidden_features' if 'hidden_features' in data else 'test_hidden_features'
    logit_key = 'logits'          if 'logits'          in data else 'test_logits'
    label_key = 'labels'          if 'labels'          in data else 'test_labels'
    return data[feat_key].float(), data[logit_key].float(), data[label_key].long()


## Load precomputed plant dataset

In [ ]:
TRAIN_DATA_PATH = '/kaggle/input/datasets/arinaromashkina/plantlet/train_data2 (1).tar.xz'
TEST_DATA_PATH  = '/kaggle/input/datasets/arinaromashkina/plantlet/test_data2.tar.xz'

train_features, train_logits, train_labels_t = load_precomputed_data(TRAIN_DATA_PATH)
test_features,  test_logits,  test_labels_t  = load_precomputed_data(TEST_DATA_PATH)

train_scores_raw = train_logits.numpy()
train_labels_raw = train_labels_t.numpy()

NUM_CLASSES = train_logits.shape[1]
FEATURE_DIM = train_features.shape[1]

print(f"Train: logits={train_logits.shape}  features={train_features.shape}")
print(f"Test:  logits={test_logits.shape}   features={test_features.shape}")
print(f"NUM_CLASSES={NUM_CLASSES}  FEATURE_DIM={FEATURE_DIM}")
print(f"Train acc={(train_scores_raw.argmax(1) == train_labels_raw).mean():.4f}")


### Dataset statistics

In [ ]:
# ── True label distribution (train + test) ───────────────────────────────────
for split, labels in [('TRAIN', train_labels_raw), ('TEST', lb_np)]:
    counts = np.bincount(labels)
    nonzero = counts[counts > 0]
    top_cls = np.argsort(counts)[::-1][:10]
    print(f"\n{split}: {len(labels)} samples, {(counts > 0).sum()} classes with data")
    print(f"  samples/class — min: {nonzero.min()}  median: {int(np.median(nonzero))}  "
          f"max: {nonzero.max()}  mean: {nonzero.mean():.1f}")
    print(f"  Top 10 classes by true label count:")
    for c in top_cls:
        if counts[c] == 0: break
        print(f"    class {c:>4d}: {counts[c]:>5d}  ({counts[c]/len(labels)*100:.1f}%)")

## Build error-conditioned decoy pools

In [ ]:
pool_score = build_error_conditioned_pools(
    train_scores_raw, train_labels_raw, NUM_CLASSES, verbose=True)


## Strategy comparison — configuration & helper functions

In [ ]:
NOISE_STD = 0.5

STRATEGIES = [
    ('score_coord',            0.0),
    ('score_coord_noise',      NOISE_STD),
    ('nearest_neighbor',       0.0),
    ('nearest_neighbor_noise', NOISE_STD),
    ('full_vector',            0.0),
    ('full_vector_noise',      NOISE_STD),
    ('binary_coord',           0.0),
    ('binary_coord_noise',     NOISE_STD),
]
STRATEGY_LABELS = {
    'score_coord':            'Random (SC)',
    'score_coord_noise':      'Random + noise',
    'nearest_neighbor':       'Nearest-neighbor',
    'nearest_neighbor_noise': 'NN + noise',
    'full_vector':            'Full vector',
    'full_vector_noise':      'Full vector + noise',
    'binary_coord':           'Binary pool',
    'binary_coord_noise':     'Binary pool + noise',
}
STRATEGY_COLORS = {
    'score_coord':            '#1976D2',
    'score_coord_noise':      '#42A5F5',
    'nearest_neighbor':       '#E65100',
    'nearest_neighbor_noise': '#FF8A65',
    'full_vector':            '#2E7D32',
    'full_vector_noise':      '#66BB6A',
    'binary_coord':           '#9C27B0',
    'binary_coord_noise':     '#CE93D8',
}

MAX_NN_POOL = 5000

def build_error_vector_pool(train_scores, train_labels, num_classes,
                            min_pool=30, max_pool_size=MAX_NN_POOL):
    """Pool of full logit vectors for NN and full_vector strategies, capped per class."""
    rng  = np.random.default_rng(0)
    pred = train_scores.argmax(axis=1)
    pool = {}
    for k in range(num_classes):
        err_mask = (pred == k) & (train_labels != k)
        vecs = train_scores[err_mask] if err_mask.sum() >= min_pool \
               else train_scores[train_labels != k]
        if len(vecs) > max_pool_size:
            vecs = vecs[rng.choice(len(vecs), size=max_pool_size, replace=False)]
        pool[k] = vecs
    print(f"  NN vector pool built: {num_classes} classes, "
          f"sizes {min(len(v) for v in pool.values())}-{max(len(v) for v in pool.values())}")
    return pool


def build_binary_pools(train_scores, train_labels, num_classes, max_pool_size=10_000):
    """Binary null pool: pool[c] = score_at_c for ALL train samples with label != c.
    This is the correct null for 'is this sample class c?'"""
    rng = np.random.default_rng(0)
    pool = {}
    for c in range(num_classes):
        neg_mask = train_labels != c
        vals = train_scores[neg_mask, c]
        if len(vals) > max_pool_size:
            vals = vals[rng.choice(len(vals), size=max_pool_size, replace=False)]
        pool[c] = vals
    sizes = [len(v) for v in pool.values()]
    print(f"  Binary pool built: {num_classes} classes, "
          f"sizes {min(sizes)}-{max(sizes)}, median={int(np.median(sizes))}")
    return pool


NN_BATCH = 1024

def _build_decoy_nearest_neighbor(sc_np, pool_error_vectors, rng, noise_std=0.0):
    """NN decoy with batched distance computation to avoid OOM."""
    n, C = sc_np.shape; pred = sc_np.argmax(1); dc = sc_np.copy(); coords = np.arange(C)
    for k in range(C):
        mask = pred == k
        if not mask.any(): continue
        pk = pool_error_vectors.get(k)
        if pk is None or len(pk) == 0: continue
        comp = coords[coords != k]
        S = sc_np[mask][:, comp]; V = pk[:, comp]; V_sq = (V ** 2).sum(1)
        best_idx = np.empty(S.shape[0], dtype=np.intp)
        for start in range(0, S.shape[0], NN_BATCH):
            end = min(start + NN_BATCH, S.shape[0])
            Sb = S[start:end]
            dists = (Sb ** 2).sum(1, keepdims=True) + V_sq[None, :] - 2 * Sb @ V.T
            best_idx[start:end] = np.maximum(dists, 0).argmin(1)
        dc[mask, k] = pk[best_idx, k]
    if noise_std > 0: dc += rng.normal(0, noise_std, dc.shape)
    return dc


def _build_decoy_full_vector(sc_np, pool_error_vectors, rng, noise_std=0.0):
    """Full vector replacement: decoy = random full error vector from pool[c_hat]."""
    pred = sc_np.argmax(axis=1); dc = np.zeros_like(sc_np)
    for k in range(sc_np.shape[1]):
        mask = pred == k
        if not mask.any(): continue
        pk = pool_error_vectors.get(k)
        if pk is None or len(pk) == 0: dc[mask] = sc_np[mask]; continue
        dc[mask] = pk[rng.choice(len(pk), size=mask.sum(), replace=True)]
    if noise_std > 0: dc += rng.normal(0, noise_std, dc.shape)
    return dc


def apply_strategy(scores_np, pool_score, pool_error_vectors, strategy, noise_std,
                   binary_pool=None):
    rng = np.random.default_rng(42)
    _n  = noise_std if strategy.endswith('_noise') else 0.0
    if strategy in ('nearest_neighbor', 'nearest_neighbor_noise'):
        return _build_decoy_nearest_neighbor(scores_np, pool_error_vectors, rng, _n)
    if strategy in ('full_vector', 'full_vector_noise'):
        return _build_decoy_full_vector(scores_np, pool_error_vectors, rng, _n)
    if strategy in ('binary_coord', 'binary_coord_noise'):
        dc = _build_decoy_score_coord(scores_np, binary_pool, rng)
        if _n > 0: dc += rng.normal(0, _n, dc.shape)
        return dc
    dc = _build_decoy_score_coord(scores_np, pool_score, rng)
    if _n > 0: dc += rng.normal(0, _n, dc.shape)
    return dc


def compute_fdr_acc_curves(scores_np, decoy_np, labels_np, pi0=0.0):
    n = len(labels_np)
    pred_sc = scores_np.max(1); pred_lb = scores_np.argmax(1); dc_sc = decoy_np.max(1)
    correct = (pred_lb == labels_np).astype(int)
    sidx = np.argsort(pred_sc); ps = pred_sc[sidx]; cs = correct[sidx]
    FD = 1 - cs; FC = np.cumsum(FD[::-1])[::-1]; DC = np.arange(n, 0, -1)
    QVAL_true = np.clip(np.minimum.accumulate(np.clip(FC / DC, 0, 1)), 0, 1)
    tsc = np.maximum(pred_sc, dc_sc); twin = (pred_sc > dc_sc).astype(int)
    ti = np.argsort(tsc); FC_t = np.cumsum((1 - twin[ti])[::-1])[::-1]
    DC_t = np.maximum(DC - FC_t, 1)
    QVAL_TDC = np.clip(np.minimum.accumulate(np.clip(FC_t / DC_t, 0, 1)), 0, 1)
    sd = np.sort(dc_sc); uz, cz = np.unique(dc_sc, return_counts=True); nuz = len(uz)
    PW = np.clip((np.searchsorted(ps, uz, 'left') - pi0 * np.searchsorted(sd, uz, 'left')) / ((1-pi0)*n), 0, 1)
    PY = np.clip(np.searchsorted(sd, uz, 'left') / n, 0, 1)
    Rj = np.clip(np.divide(PW, PY, out=np.zeros_like(PW), where=PY > 0), 0, 1)
    fdr = np.zeros(n)
    for i, T in enumerate(ps[::-1]):
        D = i + 1; F0 = pi0 * (dc_sc > T).sum()
        zi = np.searchsorted(uz, T, 'left')
        F1 = 0.0 if zi >= nuz else (1-pi0) * (Rj[zi:] * cz[zi:]).sum()
        fdr[i] = (F0 + F1) / D if D > 0 else 0
    QVAL_mm = np.clip(np.minimum.accumulate(np.clip(fdr, 0, 1)[::-1]), 0, 1)
    pi0_tdc = float(np.clip(QVAL_TDC[0], 0, 1))
    pi0_mm  = float(np.clip(QVAL_mm[0],  0, 1))
    At = np.zeros(n); Ae = np.zeros(n); Am = np.zeros(n)
    for i in range(n):
        At[i] = (cs[i:].sum() + (1 - cs[:i]).sum()) / n
        acc = n - i
        Ae[i] = np.clip((acc*(1 - QVAL_TDC[i]) + n*pi0_tdc - acc*QVAL_TDC[i]) / n, 0, 1)
        Am[i] = np.clip((acc*(1 - QVAL_mm[i])  + n*pi0_mm  - acc*QVAL_mm[i])  / n, 0, 1)
    At = np.clip(At, 0, 1)
    r  = np.arange(n) / n
    tp = int(cs.sum()); TPi = np.cumsum(cs[::-1])[::-1]; Di = DC
    acc_st = float(At[0]); acc_ta = float(At.max())
    acc_st_mm = float(Am[0]); acc_ta_mm = float(Am.max())
    return dict(
        normalized_rank=r, pred_scores_sorted=ps, pred_scores=pred_sc, decoy_scores=dc_sc,
        QVAL_true=QVAL_true, QVAL_TDC=QVAL_TDC, QVAL_mixmax=QVAL_mm,
        Acc_true=At, Acc_est=Ae, Acc_est_MM=Am,
        precision_true=np.where(Di>0, TPi/Di, 0), recall_true=TPi/max(tp,1),
        precision_est=np.clip(1-QVAL_mm, 0, 1),
        recall_est=np.clip((1-QVAL_mm)*Di/max(tp,1), 0, 1),
        correct=correct, pred_label=pred_lb, labels=labels_np,
        true_acc=float(correct.mean()), acc_st_true=acc_st, acc_ta_true=acc_ta,
        acc_st_est_mm=acc_st_mm, acc_ta_est_mm=acc_ta_mm,
        err_st_mm=abs(acc_st_mm - acc_st), err_ta_mm=abs(acc_ta_mm - acc_ta), n=n,
    )

## Build NN pool and prepare test arrays

In [ ]:
print("Building NN error vector pool...")
pool_error_vectors = build_error_vector_pool(train_scores_raw, train_labels_raw, NUM_CLASSES)

print("Building binary pools (label != c for each c)...")
binary_pool_score = build_binary_pools(train_scores_raw, train_labels_raw, NUM_CLASSES)

sc_np = test_logits.numpy()
ft_np = test_features.numpy()
lb_np = test_labels_t.numpy()

print(f"Test: n={len(lb_np)}  acc={(sc_np.argmax(1)==lb_np).mean():.4f}")
n_strats = len(STRATEGIES)

## Flow training

Train a normalizing flow for each strategy. The flow learns `P(decoy_vector | features)` from training data, then generates **all test decoys** by sampling. No raw pool decoys at test time.

Strategies:
- `score_coord` — flow trained on old error-conditioned pool targets (baseline)
- `binary_coord` — flow trained on proper binary null pool targets
- `full_vector` — flow trained on full error vector targets

In [ ]:
FLOW_EPOCHS    = 30
FLOW_LR        = 3e-4
FLOW_PATIENCE  = 5
FLOW_N         = 12
FLOW_ENC_DIM   = 128
FLOW_SUBSAMPLE = 0.5
FLOW_SEED      = 42

FLOW_STRATEGIES = [
    ('score_coord',    0.0),
    ('binary_coord',   0.0),
    ('full_vector',    0.0),
]

n_total = len(train_scores_raw)
n_flow  = int(n_total * FLOW_SUBSAMPLE)
sub_idx = np.random.default_rng(FLOW_SEED).choice(n_total, size=n_flow, replace=False)
sub_idx.sort()

flow_tr_sc = train_scores_raw[sub_idx]
flow_tr_ft = train_features.numpy()[sub_idx]
flow_tr_lb = train_labels_raw[sub_idx]
print(f"Flow subset: {n_flow}/{n_total} ({FLOW_SUBSAMPLE*100:.0f}%)"
      f"  acc={(flow_tr_sc.argmax(1)==flow_tr_lb).mean():.4f}")

sc_np = test_logits.numpy()
ft_np = test_features.numpy()
lb_np = test_labels_t.numpy()
print(f"Test: n={len(lb_np)}  acc={(sc_np.argmax(1)==lb_np).mean():.4f}")

# Train flows and generate test decoys
flow_results = {}      # strat -> curves dict
flow_decoys  = {}      # strat -> (model_scores [N,C], decoy_scores [N,C], labels [N])

for strat_name, noise_std in FLOW_STRATEGIES:
    print(f'\n{"#"*55}  {strat_name}')
    train_decoy = apply_strategy(flow_tr_sc, pool_score, pool_error_vectors, strat_name, noise_std,
                                 binary_pool=binary_pool_score)
    train_ds = ScoreFeatureDataset(
        torch.from_numpy(flow_tr_sc).float(), torch.from_numpy(flow_tr_ft).float(),
        torch.from_numpy(train_decoy).float(), torch.from_numpy(flow_tr_lb).long())

    flow_path = f'plant_flow_{strat_name}_half.pth'
    flow = ScoreShiftFlowWrapper(NUM_CLASSES, FLOW_N, FEATURE_DIM, 256, FLOW_ENC_DIM, 5.0).to(DEVICE)

    if os.path.exists(flow_path):
        flow.load_state_dict(torch.load(flow_path, map_location=DEVICE, weights_only=False))
        print(f"  Loaded from {flow_path}")
    else:
        print(f"  Training -> {flow_path}")
        flow.train_flow(train_ds, epochs=FLOW_EPOCHS, lr=FLOW_LR,
                        batch_size=256, device=str(DEVICE),
                        patience=FLOW_PATIENCE, grad_clip=1.0)
        torch.save(flow.state_dict(), flow_path)
        print(f"  Saved -> {flow_path}")

    flow.eval()
    test_ds = ScoreFeatureDataset(
        torch.from_numpy(sc_np).float(), torch.from_numpy(ft_np).float(),
        torch.from_numpy(sc_np).float(),  # placeholder
        torch.from_numpy(lb_np).long())
    ms_np, ds_np, ls_np = flow.generate_decoys(test_ds, device=str(DEVICE))

    flow_decoys[strat_name] = (ms_np, ds_np, ls_np)
    crv = compute_fdr_acc_curves(ms_np, ds_np, ls_np)
    flow_results[strat_name] = crv
    print(f"  true_acc={crv['true_acc']:.3f}  err_st={crv['err_st_mm']:.3f}  err_ta={crv['err_ta_mm']:.3f}")

print('\nFlow experiments done.')
n_strats = len(FLOW_STRATEGIES)

### Flow decoys: FDR and Accuracy curves

In [ ]:
fig, axes = plt.subplots(2, n_strats, figsize=(6*n_strats, 9))
if n_strats == 1: axes = axes.reshape(2, 1)

for col, (strat_name, _) in enumerate(FLOW_STRATEGIES):
    fc = flow_results[strat_name]; r = fc['normalized_rank']
    color = STRATEGY_COLORS[strat_name]

    ax = axes[0][col]
    ax.plot(r, fc['QVAL_true'],   'k--', lw=2.5, label='True FDR')
    ax.plot(r, fc['QVAL_mixmax'], color=color, lw=2, label=f'MixMax  err_st={fc["err_st_mm"]:.3f}')
    ax.plot(r, fc['QVAL_TDC'],   color='navy', lw=1.2, ls=':', label='TDC')
    ax.axhline(0.1, color='black', lw=0.7, ls=':', alpha=0.4)
    ax.set_title(STRATEGY_LABELS[strat_name], fontsize=12)
    ax.set_xlabel('Fraction accepted'); ax.set_ylim(0, 1.05)
    ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)
    if col == 0: ax.set_ylabel('q-value (FDR)')

    ax = axes[1][col]
    ax.plot(r, fc['Acc_true'],   'k--', lw=2.5, label='True Acc')
    ax.plot(r, fc['Acc_est_MM'], color=color, lw=2, label=f'MixMax  err_ta={fc["err_ta_mm"]:.3f}')
    ax.set_xlabel('Fraction accepted')
    ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)
    if col == 0: ax.set_ylabel('Accuracy')

plt.suptitle('Plant — Flow decoys: FDR and Accuracy', fontsize=14)
plt.tight_layout(); plt.show()

### Flow decoys: max(model) vs max(decoy) scatter

In [ ]:
corr_mask = (sc_np.argmax(1) == lb_np)
fig, axes = plt.subplots(1, n_strats, figsize=(6*n_strats, 5))
if n_strats == 1: axes = [axes]

for ax, (strat_name, _) in zip(axes, FLOW_STRATEGIES):
    ms, ds, ls = flow_decoys[strat_name]
    ms_max = ms.max(1); ds_max = ds.max(1)
    idx = np.random.default_rng(0).choice(len(ms_max), min(4000, len(ms_max)), replace=False)
    cor = corr_mask[idx]
    ax.scatter(ms_max[idx][cor],  ds_max[idx][cor],  s=5, alpha=0.3, color='steelblue',
               label='correct', rasterized=True)
    ax.scatter(ms_max[idx][~cor], ds_max[idx][~cor], s=5, alpha=0.5, color='crimson',
               label='incorrect', rasterized=True)
    lims = [min(ms_max.min(), ds_max.min())-0.2, max(ms_max.max(), ds_max.max())+0.2]
    ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.5)
    ax.set_xlabel('max(model)'); ax.set_ylabel('max(flow decoy)')
    ax.set_title(STRATEGY_LABELS[strat_name]); ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

plt.suptitle('Plant — Flow decoys: max(model) vs max(decoy)', fontsize=13)
plt.tight_layout(); plt.show()

### Flow decoys: score distributions

In [ ]:
ref = flow_results[FLOW_STRATEGIES[0][0]]
bins = np.linspace(ref['pred_scores'].min()-0.3, ref['pred_scores'].max()+0.3, 60)
inc_mask = ref['correct'] == 0

fig, axes = plt.subplots(1, n_strats, figsize=(6*n_strats, 4.5))
if n_strats == 1: axes = [axes]
for ax, (strat_name, _) in zip(axes, FLOW_STRATEGIES):
    fc = flow_results[strat_name]; color = STRATEGY_COLORS[strat_name]
    sns.histplot(fc['pred_scores'],  bins=bins, stat='density', color='steelblue',
                 kde=True, fill=True, alpha=0.3, label='model', ax=ax)
    sns.histplot(fc['decoy_scores'], bins=bins, stat='density', color=color,
                 kde=True, fill=True, alpha=0.4, label='flow decoy', ax=ax)
    if inc_mask.any():
        sns.histplot(fc['pred_scores'][inc_mask], bins=bins, stat='density', color='crimson',
                     kde=True, fill=True, alpha=0.2, label='incorrect', ax=ax)
    ax.set_title(f'{STRATEGY_LABELS[strat_name]}\nacc={fc["true_acc"]:.3f}', fontsize=11)
    ax.set_xlabel('Max logit'); ax.legend(fontsize=7)
plt.suptitle('Plant — Flow decoy score distributions', fontsize=13)
plt.tight_layout(); plt.show()

### All flow strategies: overlay comparison

In [ ]:
r = flow_results[FLOW_STRATEGIES[0][0]]['normalized_rank']
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(r, flow_results[FLOW_STRATEGIES[0][0]]['Acc_true'], 'k--', lw=2.5, label='True Acc')
for strat_name, _ in FLOW_STRATEGIES:
    fc = flow_results[strat_name]
    ax.plot(r, fc['Acc_est_MM'], color=STRATEGY_COLORS[strat_name], lw=2,
            label=f'{STRATEGY_LABELS[strat_name]} ({fc["err_st_mm"]:.3f})')
ax.set_xlabel('Fraction accepted'); ax.set_ylabel('Accuracy')
ax.set_title('Accuracy — all flow strategies'); ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.4)

ax = axes[1]
ax.plot(r, flow_results[FLOW_STRATEGIES[0][0]]['QVAL_true'], 'k--', lw=2.5, label='True FDR')
for strat_name, _ in FLOW_STRATEGIES:
    fc = flow_results[strat_name]
    ax.plot(r, fc['QVAL_mixmax'], color=STRATEGY_COLORS[strat_name], lw=2,
            label=STRATEGY_LABELS[strat_name])
ax.set_xlabel('Fraction accepted'); ax.set_ylabel('q-value'); ax.set_ylim(0, 1.05)
ax.set_title('FDR — all flow strategies'); ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.4)

plt.suptitle('Plant — Flow strategies comparison', fontsize=14)
plt.tight_layout(); plt.show()

### MAE summary — flow decoys

In [ ]:
print('='*60)
print('MAE SUMMARY — Flow Decoys')
print('='*60)
print(f"  {'Strategy':<25} {'err_ST':>8}  {'err_TA':>8}")
print(f"  {'-'*45}")
for strat_name, _ in FLOW_STRATEGIES:
    fc = flow_results[strat_name]
    print(f"  {STRATEGY_LABELS[strat_name]:<25} {fc['err_st_mm']:>8.4f}  {fc['err_ta_mm']:>8.4f}")

### Per-class binary FDR from flow decoys — class 204

Take the flow-generated decoy vectors (full C-dim) and evaluate as binary for one class:
- Score = `logit[:, c]`, Decoy = `flow_decoy[:, c]`
- Positive = `label == c`, all 31112 samples
- FDR via MixMax (from flow decoy at coord c) and BH (from binary pool directly)

This shows whether the flow learned a good null distribution at each coordinate.

In [ ]:
TARGET_C = 204
STRAT_FOR_BINARY = 'binary_coord'

ms_full, ds_full, ls_full = flow_decoys[STRAT_FOR_BINARY]
model_at_c = ms_full[:, TARGET_C]
decoy_at_c = ds_full[:, TARGET_C]
is_pos     = (ls_full == TARGET_C)
n = len(model_at_c)

print(f"Binary FDR from flow decoys ({STRAT_FOR_BINARY}) — class {TARGET_C}")
print(f"  n={n}  pos={is_pos.sum()}  neg={(~is_pos).sum()}")
print(f"  model at {TARGET_C}: pos med={np.median(model_at_c[is_pos]):.2f}  neg med={np.median(model_at_c[~is_pos]):.2f}")
print(f"  FLOW decoy at {TARGET_C}: pos med={np.median(decoy_at_c[is_pos]):.2f}  neg med={np.median(decoy_at_c[~is_pos]):.2f}")

# Sort by model_at_c
sidx = np.argsort(model_at_c)
ms_s = model_at_c[sidx]; cs_s = is_pos[sidx].astype(int)
DC = np.arange(n, 0, -1)

# True FDR
FD_true = np.cumsum((1 - cs_s)[::-1])[::-1]
QVAL_true = np.clip(np.minimum.accumulate(np.clip(FD_true / DC, 0, 1)), 0, 1)

# MixMax from flow decoy at coord c
ds_c = decoy_at_c
sd = np.sort(ds_c); uz, cz = np.unique(ds_c, return_counts=True); nuz = len(uz)
PW = np.clip(np.searchsorted(ms_s, uz, 'left') / n, 0, 1)
PY = np.clip(np.searchsorted(sd, uz, 'left') / n, 0, 1)
Rj = np.clip(np.divide(PW, PY, out=np.zeros_like(PW), where=PY > 0), 0, 1)
fdr_mm = np.zeros(n)
for i, T in enumerate(ms_s[::-1]):
    D = i + 1; zi = np.searchsorted(uz, T, 'left')
    fdr_mm[i] = (0.0 if zi >= nuz else (Rj[zi:] * cz[zi:]).sum()) / D if D > 0 else 0
QVAL_mm = np.clip(np.minimum.accumulate(np.clip(fdr_mm, 0, 1)[::-1]), 0, 1)

# TDC from flow decoy at coord c
tsc = np.maximum(model_at_c, ds_c)
twin = (model_at_c > ds_c).astype(int)
ti = np.argsort(tsc)
FC_t = np.cumsum((1 - twin[ti])[::-1])[::-1]
DC_t = np.maximum(DC - FC_t, 1)
QVAL_TDC = np.clip(np.minimum.accumulate(np.clip(FC_t / DC_t, 0, 1)), 0, 1)

# BH from binary pool (for comparison — not flow, direct pool)
pool_c = binary_pool_score[TARGET_C]
pool_sorted = np.sort(pool_c)
pv = np.clip(1.0 - np.searchsorted(pool_sorted, model_at_c, side='right') / len(pool_sorted),
             1.0 / len(pool_sorted), 1.0)
si_bh = np.argsort(pv)
qv_bh = np.zeros(n)
qv_sorted = np.minimum.accumulate((pv[si_bh] * n / np.arange(1, n + 1))[::-1])[::-1]
qv_bh[si_bh] = np.clip(qv_sorted, 0, 1)
bh_sorted = qv_bh[sidx]

r = np.arange(n) / n

# ── PLOTS ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# (0,0) Scatter: model vs flow decoy at coord c
ax = axes[0][0]
ax.scatter(model_at_c[~is_pos], decoy_at_c[~is_pos],
           s=2, alpha=0.1, color='gray', label=f'neg ({(~is_pos).sum()})', rasterized=True)
ax.scatter(model_at_c[is_pos],  decoy_at_c[is_pos],
           s=8, alpha=0.6, color='crimson', label=f'pos ({is_pos.sum()})', rasterized=True)
lo = min(model_at_c.min(), decoy_at_c.min()) - 0.5
hi = max(model_at_c.max(), decoy_at_c.max()) + 0.5
ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8, alpha=0.5)
ax.set_xlabel(f'Model logit at {TARGET_C}'); ax.set_ylabel(f'Flow decoy at {TARGET_C}')
ax.set_title(f'Flow decoy at coord {TARGET_C}\n(flow generates full vector, we slice coord {TARGET_C})')
ax.legend(fontsize=8, markerscale=2); ax.grid(ls='--', alpha=0.3)

# (0,1) Distributions
ax = axes[0][1]
bins_b = np.linspace(lo, hi, 80)
ax.hist(model_at_c[is_pos],  bins=bins_b, density=True, alpha=0.5, color='crimson', label='pos model')
ax.hist(model_at_c[~is_pos], bins=bins_b, density=True, alpha=0.2, color='gray', label='neg model')
ax.hist(decoy_at_c, bins=bins_b, density=True, histtype='step', lw=2, color='#9C27B0',
        label='flow decoy (all)')
ax.hist(pool_c, bins=bins_b, density=True, histtype='step', lw=1.5, ls='--', color='orange',
        label='binary pool')
ax.set_xlabel(f'Logit at {TARGET_C}'); ax.set_ylabel('Density')
ax.set_title('Distributions at coord c'); ax.legend(fontsize=7); ax.grid(ls='--', alpha=0.3)

# (0,2) p-value histogram (BH from pool)
ax = axes[0][2]
bins_p = np.linspace(0, 1, 31)
ax.hist(pv[~is_pos], bins=bins_p, density=True, alpha=0.5, color='#2E7D32',
        label=f'Neg (mean={pv[~is_pos].mean():.2f})')
ax.hist(pv[is_pos],  bins=bins_p, density=True, alpha=0.5, color='crimson',
        label=f'Pos (mean={pv[is_pos].mean():.2f})')
ax.axhline(1, ls='--', color='black', lw=1)
ax.set_xlabel('p-value (from binary pool)'); ax.set_ylabel('Density')
ax.set_title('BH p-values'); ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

# (1,0) FDR curves
ax = axes[1][0]
ax.plot(r, QVAL_true,   'k--', lw=2.5, label='True FDR')
ax.plot(r, QVAL_mm,     color='#9C27B0', lw=2,   label='MixMax (flow decoy)')
ax.plot(r, QVAL_TDC,    color='navy',    lw=1.2, ls=':', label='TDC (flow decoy)')
ax.plot(r, bh_sorted,   color='#2E7D32', lw=1.5, ls='-.', label='BH (binary pool)')
ax.axhline(0.1, color='black', lw=0.7, ls=':', alpha=0.4)
ax.set_xlabel('Fraction accepted'); ax.set_ylabel('q-value'); ax.set_ylim(0, 1.05)
ax.set_title(f'Binary FDR — class {TARGET_C}')
ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

# (1,1) Zoom top 10%
ax = axes[1][1]
zoom = r > 0.9
ax.plot(r[zoom], QVAL_true[zoom],  'k--', lw=2.5, label='True FDR')
ax.plot(r[zoom], QVAL_mm[zoom],    color='#9C27B0', lw=2, label='MixMax (flow)')
ax.plot(r[zoom], bh_sorted[zoom],  color='#2E7D32', lw=1.5, ls='-.', label='BH (pool)')
ax.set_xlabel('Fraction accepted'); ax.set_ylabel('q-value'); ax.set_ylim(0, 1.05)
ax.set_title('ZOOM: top 10%'); ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

# (1,2) Precision-Recall
ax = axes[1][2]
tp_cum = np.cumsum(cs_s[::-1])[::-1]
prec_true = tp_cum / DC; rec_true = tp_cum / max(is_pos.sum(), 1)
prec_mm = np.clip(1 - QVAL_mm, 0, 1); rec_mm = prec_mm * DC / max(is_pos.sum(), 1)
prec_bh = np.clip(1 - bh_sorted, 0, 1); rec_bh = prec_bh * DC / max(is_pos.sum(), 1)
ax.plot(rec_true, prec_true, 'k--', lw=2.5, label='True PR')
ax.plot(rec_mm, prec_mm, color='#9C27B0', lw=2, label='MixMax (flow)')
ax.plot(rec_bh, prec_bh, color='#2E7D32', lw=1.5, ls='-.', label='BH (pool)')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_xlim(0, 1.05); ax.set_ylim(0, 1.05)
ax.set_title(f'PR — class {TARGET_C}'); ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.3)

plt.suptitle(f'Binary FDR from flow decoys ({STRAT_FOR_BINARY}) — class {TARGET_C}\n'
             f'n={n}  pos={is_pos.sum()}  (flow generates full vector, sliced at coord {TARGET_C})',
             fontsize=13)
plt.tight_layout(); plt.show()

# Summary
print(f"\nDiscoveries at controlled FDR (class {TARGET_C}):")
print(f"  {'alpha':<7} {'MM(flow)':>12} {'TDC(flow)':>12} {'BH(pool)':>12}  (/{is_pos.sum()} pos)")
print(f"  {'-'*52}")
for alpha in [0.01, 0.05, 0.10, 0.20]:
    mm_m = QVAL_mm <= alpha; mm_d = mm_m.sum(); mm_tp = cs_s[mm_m].sum()
    td_m = QVAL_TDC <= alpha; td_d = td_m.sum(); td_tp = cs_s[td_m].sum() if td_d else 0
    bh_d = (qv_bh <= alpha).sum(); bh_tp = (qv_bh[is_pos] <= alpha).sum()
    print(f"  {alpha:<7.2f} {mm_d:>5}({mm_tp:>3}tp) {td_d:>5}({td_tp:>3}tp) {bh_d:>5}({bh_tp:>3}tp)")